In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('test2') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/09 15:33:50 WARN Utils: Your hostname, codespaces-00ce04, resolves to a loopback address: 127.0.0.1; using 10.0.10.234 instead (on interface eth0)
26/11/09 15:33:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 15:33:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

26/11/09 15:33:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

In [3]:
df_green.registerTempTable('green')

/workspaces/spark-learn/venv/lib/python3.14/site-packages/pyspark/sql/classic/dataframe.py:186: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [8]:
df_green_revenue = spark.sql("""
SELECT 
    PULocationID AS zone,
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE 
    lpep_pickup_datetime >= '2021-01-01'
GROUP BY 
    1,2
ORDER BY 
    1,2
                             """)

In [9]:
df_green_revenue.show()

+----+-------------------+------+--------------+
|zone|               hour|amount|number_records|
+----+-------------------+------+--------------+
|   1|2021-01-29 09:00:00| 100.3|             1|
|   1|2021-03-02 18:00:00| 115.3|             1|
|   1|2021-03-12 06:00:00|  90.3|             1|
|   1|2021-04-10 19:00:00| 131.3|             1|
|   1|2021-04-12 07:00:00|  92.8|             1|
|   1|2021-06-08 07:00:00| 101.1|             1|
|   1|2021-06-29 06:00:00|125.16|             1|
|   1|2021-08-04 19:00:00|  80.3|             1|
|   1|2021-08-11 06:00:00|  85.3|             1|
|   1|2021-09-03 07:00:00| 74.55|             1|
|   1|2021-10-02 06:00:00|112.88|             1|
|   1|2021-11-12 10:00:00|  70.3|             1|
|   1|2021-11-18 08:00:00|110.76|             1|
|   1|2021-11-28 19:00:00|144.36|             1|
|   1|2021-11-30 07:00:00| 130.3|             1|
|   1|2021-12-08 16:00:00|  51.3|             1|
|   1|2021-12-19 07:00:00|  95.3|             1|
|   1|2021-12-25 17: